# **0. Import libraries**

In [1]:
#Import the libraries for collecting the related data from a given webpage

import time
import random
from datetime import datetime,date
import pandas as pd
import os
from io import StringIO
import glob

import asyncio #For asynchronous data collection, which boosts up the collection speed of the data
import aiohttp #For asynchronous web scraping
import nest_asyncio #For running asynchronous functions in jupyter notebook


# **1. Check if the data is already collected**

In [2]:
#Check if there is an existing data collected from the web

DATA_NAME = "BoxOffice_Final.csv"

DATA_EXISTENCE = True

if DATA_NAME in os.listdir("./"):
    pass
else:
    DATA_EXISTENCE = False

# **2. Create web scrapping functions**

In [3]:
#Creating a web scrapper

MAX_REQUESTS = 4 #Setting a threshold for the number of requests thrown at the same time
MAX_RETRIES = 3 #Number of retries for the data collection

#Make a function that scrapes a data at a given timeline
async def scrape_single_date(session, date_str, semaphore):
     
    url = f"https://www.boxofficemojo.com/date/{date_str}/" #Boxofficemojo has this structure of url that date_str is in url to look up for data at specific time
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    async with semaphore:
        for attempt in range(MAX_RETRIES):
            await asyncio.sleep(random.uniform(1.5, 2.5))

            try:
                async with session.get(url, headers=headers, timeout=10) as response:
                    #When we get the response from the website, collect the data

                    if response.status in [503, 429]:
                        wait_time = (attempt + 1) * random.uniform(1, 3)
                        print(f"Server busy ({response.status}) for {date_str}. Retrying in {wait_time:.1f}s, (Attempt {attempt+1}/{MAX_RETRIES})")
                        await asyncio.sleep(wait_time)
                        continue

                    if response.status == 200:
                        html_text = await response.text()

                        #Read the information from the given data
                        dfs = pd.read_html(StringIO(html_text))

                        if dfs and len(dfs) > 0:
                            daily_table = dfs[0]

                            #Collect the information of total revenue
                            daily_clean_str = daily_table['Daily'].astype(str).str.replace('$', '').str.replace(',', '').astype(float)
                            daily_table['Daily_clean'] = pd.to_numeric(daily_clean_str, errors='coerce').fillna(0)

                            #Collect the information of total theatres which put the movie on the screen
                            theaters_clean_str = daily_table['Theaters'].astype(str).str.replace(',', '', regex=False)
                            daily_table['Theaters_clean'] = pd.to_numeric(theaters_clean_str, errors='coerce').fillna(0)

                            #Collect a few more variables

                            #Total number of revenue on the day
                            total_gross = daily_table['Daily_clean'].sum()

                            #Rank variable showing if a given movie was the one with the most revenue
                            top_movie_gross = daily_table['Daily_clean'].max() if not daily_table.empty else 0 

                            #Total number of theatres which put the movie on the screen
                            total_theaters = daily_table['Theaters_clean'].sum()

                            #Total number of movies on screen on the day
                            movie_list = list(daily_table['Release'])

                            print(f"The data for the date: {date_str} collected with {len(daily_table)} entries")

                            return {
                                        'Date': date_str,
                                        'Total_Gross': total_gross,
                                        'Top_Movie_Gross': top_movie_gross,
                                        'Total_Theaters': total_theaters,
                                        'Movie_list': movie_list
                                    }
                    #There might be a case when the server does not respond, make exception for that
                    else:

                        print(f"Error on the day: {date_str}, error code {response.status}")
                        return {'Date': date_str, 'Status': f'Response_status: {response.status}'}
                    
            #There might be a case when there are other exceptions by running the script, make exception for that
            except Exception as e:

                if attempt < MAX_RETRIES - 1:
                    wait_time = (attempt + 1) * 3
                    print(f"Connection error for {date_str}: {str(e)}. Retrying in {wait_time}s")
                    await asyncio.sleep(wait_time)
                    continue

                print(f"There is an exception occurred on the day {date_str}: {str(e)}")
                return {'Date': date_str, 'Status': f'Error: {e}'}

        #Add a case when all retries failed
        print(f"Failed to collect data for {date_str} after {MAX_RETRIES} attempts due to server throttling.")
        return {'Date': date_str, 'Status': 'Failed after maximum retries'}

#Make another function which takes in multiple date that uses 

async def main_scraper(start_date_str, end_date_str):
    date_range = pd.date_range(start=start_date_str, end=end_date_str, freq='D')
    date_strs = [d.strftime('%Y-%m-%d') for d in date_range]
    
    semaphore = asyncio.Semaphore(MAX_REQUESTS)
    
    #Use 1 client session for multiple dates
    async with aiohttp.ClientSession() as session:
        #Create tasks for multiple days
        tasks = [scrape_single_date(session, date_str, semaphore) for date_str in date_strs]
        
        #Gather the results together
        results = await asyncio.gather(*tasks)
        
    return pd.DataFrame(results)


# **3. Run the scrapping function and collect the data from boxofficemojo.com**

In [ ]:
#Run the first data collection script

nest_asyncio.apply() #Initiate async for notebook

#If the csv data doesn't exist, start scrapping
if DATA_EXISTENCE == False:

    #Initialise the data collection with start and end year, to collect data year by year
    start_year = 2022
    end_year = 2026

    #For loop for jupyter notebook
    loop = asyncio.get_event_loop()

    #Start collecting for each year
    for year in range(start_year, end_year+1):

        start_time = time.time() #To measure the total amount of time for data collection
        
        #Dataframe for collecting each month's data
        monthly_dfs = []

        max_month = 6 if year == end_year else 12

        print(f"\n========== Start Scrapping for Year: {year} (Splitting into months) ==========")

        for month in range(1, max_month + 1):
            #Compute monthly start
            month_start = date(year, month, 1)
            
            #Compute monthly end
            if month in [4, 6, 9, 11]:
                month_end = date(year, month, 30)
            elif month == 2:
                #Compute leap year
                is_leap = (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)
                month_end = date(year, month, 29 if is_leap else 28)
            else:
                month_end = date(year, month, 31)
                
            print(f" -> Collecting: {month_start.strftime('%Y-%m-%d')} ~ {month_end.strftime('%Y-%m-%d')}")
            
            #Run month by month
            df_month = loop.run_until_complete(main_scraper(month_start, month_end))

            #Stack up the monthly data
            if df_month is not None and not df_month.empty:
                monthly_dfs.append(df_month)
                
            #Give break in between data collection for months
            time.sleep(random.uniform(3.0, 5.0))
            
        #Save a yearly record after yearly collection is done
        if monthly_dfs:
            df_boxoffice = pd.concat(monthly_dfs, ignore_index=True)
            df_boxoffice.to_csv(f"BoxOffice_{year}.csv", index=False, encoding='utf-8')
            print(f"Data for year: {year} has been saved with {len(df_boxoffice)} rows")
            print(f"Total amount of time taken for data collection: {time.time() - start_time:.2f} seconds")
        else:
            print(f"No data collected for year: {year}")

        if year < end_year:
                print(f"Getting into a break time after scrapping for the year: {year}")
                time.sleep(10)

    #Import all collected data csv
    all_files = glob.glob("BoxOffice_*.csv")
    combined_df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

    #Merge them into one
    combined_df = combined_df.sort_values(by="Date")
    combined_df.to_csv(f"{DATA_NAME}", index=False, encoding='utf-8')

#If the csv exists, pass this scrapping part
else:
    print("CSV data already exists. Skipping data collection.")


========== Start Scrapping for Year: 2022 (Splitting into months) ==========
 -> Collecting: 2022-01-01 ~ 2022-01-31
The data for the date: 2022-01-03 collected with 21 entries
The data for the date: 2022-01-01 collected with 26 entries
The data for the date: 2022-01-04 collected with 23 entries
The data for the date: 2022-01-02 collected with 25 entries
The data for the date: 2022-01-05 collected with 21 entries
The data for the date: 2022-01-07 collected with 25 entries
The data for the date: 2022-01-06 collected with 21 entries
The data for the date: 2022-01-08 collected with 25 entries
The data for the date: 2022-01-09 collected with 25 entries
The data for the date: 2022-01-10 collected with 24 entries
The data for the date: 2022-01-12 collected with 24 entries
The data for the date: 2022-01-11 collected with 24 entries
The data for the date: 2022-01-15 collected with 25 entries
The data for the date: 2022-01-13 collected with 24 entries
The data for the date: 2022-01-14 collecte